In [ ]:
import nltk
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
documents = [
    """The scientist adjusted the light in the laboratory to study the specimen more clearly.
       She measured the wavelength of visible light using a spectrometer.
       The experiment required precise light conditions to yield accurate results.""",

    """After a long journey, the traveler felt light as he dropped his heavy backpack.
       He sat by the window where the morning light gently warmed his face.
       His mood was light and cheerful as he sipped his tea.""",

    """The firefighter used a light to navigate through the dark, smoke-filled corridor.
       His equipment was surprisingly light, allowing quick movement.
       He finally saw a faint light at the end of the hallway signaling the exit."""
]

print("THREE DOCUMENTS WITH AMBIGUOUS 'LIGHT'")
print()
for i, doc in enumerate(documents, 1):
    print(f"\nDocument {i}:\n{doc}")

THREE DOCUMENTS WITH AMBIGUOUS 'LIGHT'


Document 1:
The scientist adjusted the light in the laboratory to study the specimen more clearly. 
       She measured the wavelength of visible light using a spectrometer. 
       The experiment required precise light conditions to yield accurate results.

Document 2:
After a long journey, the traveler felt light as he dropped his heavy backpack. 
       He sat by the window where the morning light gently warmed his face. 
       His mood was light and cheerful as he sipped his tea.

Document 3:
The firefighter used a light to navigate through the dark, smoke-filled corridor. 
       His equipment was surprisingly light, allowing quick movement. 
       He finally saw a faint light at the end of the hallway signaling the exit.


In [ ]:
print("TF-IDF MATRIX")

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(documents)
feature_names = vectorizer.get_feature_names_out()

# Display TF-IDF scores per document
for i, doc in enumerate(documents, 1):
    print(f"\n--- Document {i} Top TF-IDF Terms ---")
    tfidf_scores = tfidf_matrix[i-1].toarray()[0]
    top_indices = tfidf_scores.argsort()[-8:][::-1]
    for idx in top_indices:
        if tfidf_scores[idx] > 0:
            print(f"  {feature_names[idx]:<20} : {tfidf_scores[idx]:.4f}")

TF-IDF MATRIX

--- Document 1 Top TF-IDF Terms ---
  light                : 0.3854
  wavelength           : 0.2175
  yield                : 0.2175
  using                : 0.2175
  specimen             : 0.2175
  spectrometer         : 0.2175
  visible              : 0.2175
  results              : 0.2175

--- Document 2 Top TF-IDF Terms ---
  light                : 0.3948
  warmed               : 0.2228
  window               : 0.2228
  sat                  : 0.2228
  sipped               : 0.2228
  tea                  : 0.2228
  traveler             : 0.2228
  journey              : 0.2228

--- Document 3 Top TF-IDF Terms ---
  light                : 0.3766
  smoke                : 0.2125
  used                 : 0.2125
  signaling            : 0.2125
  saw                  : 0.2125
  surprisingly         : 0.2125
  quick                : 0.2125
  faint                : 0.2125


In [ ]:
print("COSINE SIMILARITY MATRIX")

cosine_sim = cosine_similarity(tfidf_matrix)

print("\n         Doc1    Doc2    Doc3")
print("         ------  ------  ------")
for i in range(3):
    row = f"Doc{i+1}  |"
    for j in range(3):
        row += f"  {cosine_sim[i][j]:.4f}"
    print(row)

COSINE SIMILARITY MATRIX

         Doc1    Doc2    Doc3
         ------  ------  ------
Doc1  |  1.0000  0.1522  0.1451
Doc2  |  0.1522  1.0000  0.1487
Doc3  |  0.1451  0.1487  1.0000


In [ ]:
print("QUERY EXPANSION WITH WORDNET SYNONYMS")

def expand_query(query):
    """Expand query by adding WordNet synonyms for each token."""
    tokens = query.lower().split()
    expanded = []

    for word in tokens:
        expanded.append(word)
        synsets = wordnet.synsets(word)
        for syn in synsets:
            for lemma in syn.lemmas():
                synonym = lemma.name().replace('_', ' ')
                expanded.append(synonym)

    return list(dict.fromkeys(expanded))  # Remove duplicates


# Original ambiguous query
original_query = "light"
expanded_terms  = expand_query(original_query)

print(f"\nOriginal Query   : '{original_query}'")
print(f"Expanded Terms   : {expanded_terms[:15]}...")
print(f"Total Synonyms   : {len(expanded_terms)}")


QUERY EXPANSION WITH WORDNET SYNONYMS

Original Query   : 'light'
Expanded Terms   : ['light', 'visible light', 'visible radiation', 'light source', 'luminosity', 'brightness', 'brightness level', 'luminance', 'luminousness', 'illumination', 'lightness', 'lighting', 'sparkle', 'twinkle', 'spark']...
Total Synonyms   : 61


In [ ]:
print("IR RANKING WITH ORIGINAL vs EXPANDED QUERY")

def get_rankings(query, documents, vectorizer, tfidf_matrix, label):
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    print(f"\n  [{label}] Query: '{query[:60]}'")
    ranked = sorted(enumerate(similarities, 1), key=lambda x: x[1], reverse=True)
    for rank, (doc_id, score) in enumerate(ranked, 1):
        print(f"    Rank {rank} → Document {doc_id}  |  Cosine Score: {score:.4f}")
    return ranked

# Original query ranking
get_rankings("light", documents, vectorizer, tfidf_matrix, "ORIGINAL QUERY")

# Expanded query ranking
expanded_query_str = " ".join(expanded_terms[:10])
get_rankings(expanded_query_str, documents, vectorizer, tfidf_matrix, "EXPANDED QUERY")

print()
print("  CONCLUSION: Expanded query improves document retrieval by capturing different senses of the ambiguous word.")

IR RANKING WITH ORIGINAL vs EXPANDED QUERY

  [ORIGINAL QUERY] Query: 'light'
    Rank 1 → Document 2  |  Cosine Score: 0.3948
    Rank 2 → Document 1  |  Cosine Score: 0.3854
    Rank 3 → Document 3  |  Cosine Score: 0.3766

  [EXPANDED QUERY] Query: 'light visible light visible radiation light source luminosit'
    Rank 1 → Document 1  |  Cosine Score: 0.4183
    Rank 2 → Document 2  |  Cosine Score: 0.2618
    Rank 3 → Document 3  |  Cosine Score: 0.2497

  CONCLUSION: Expanded query improves document retrieval by capturing different senses of the ambiguous word.
